**Вариант 10**

**Практическая работа 2**

Исходные данные

In [19]:
import pandas as pd
import numpy as np

# таблица продаж с заказами, товарами, количеством, ценой и регионом
sales = pd.DataFrame({
    'OrderID': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'Product': ['Laptop', 'Phone', 'Tablet', 'Laptop', 'Phone',
                'Laptop', 'Tablet', 'Phone', 'Laptop', 'Tablet'],
    'Quantity': [1, 2, 1, 3, 1, 2, 1, 4, 1, 2],
    'Price': [50000, 30000, 20000, 50000, np.nan,
              50000, 20000, 30000, 50000, np.nan],
    'Region': ['Moscow', 'SPb', 'Kazan', 'Moscow', 'SPb',
               'Kazan', np.nan, 'SPb', 'Moscow', 'Kazan']
})

# справочник товаров с категориями и весом
products = pd.DataFrame({
    'Product': ['Laptop', 'Phone', 'Tablet', 'Monitor'],
    'Category': ['Electronics', 'Electronics', 'Electronics', 'Accessories'],
    'Weight_kg': [2.5, 0.2, 0.5, 3.0]
})

# выводим обе таблицы для просмотра исходных данных
print('Таблица продаж:')
print(sales)
print('\nТаблица товаров:')
print(products)


Таблица продаж:
   OrderID Product  Quantity    Price  Region
0      101  Laptop         1  50000.0  Moscow
1      102   Phone         2  30000.0     SPb
2      103  Tablet         1  20000.0   Kazan
3      104  Laptop         3  50000.0  Moscow
4      105   Phone         1      NaN     SPb
5      106  Laptop         2  50000.0   Kazan
6      107  Tablet         1  20000.0     NaN
7      108   Phone         4  30000.0     SPb
8      109  Laptop         1  50000.0  Moscow
9      110  Tablet         2      NaN   Kazan

Таблица товаров:
   Product     Category  Weight_kg
0   Laptop  Electronics        2.5
1    Phone  Electronics        0.2
2   Tablet  Electronics        0.5
3  Monitor  Accessories        3.0


**1. Найти продукт с минимальной средней ценой**

In [20]:
avg_price_by_product = sales.groupby('Product')['Price'].mean()
print(avg_price_by_product)
print(f'\nПродукт с минимальной средней ценой: {avg_price_by_product.idxmin()}')
print(f'Средняя цена: {avg_price_by_product.min()}')


Product
Laptop    50000.0
Phone     30000.0
Tablet    20000.0
Name: Price, dtype: float64

Продукт с минимальной средней ценой: Tablet
Средняя цена: 20000.0


**2. Посчитать выручку для каждого заказа и найти топ-5 по выручке**

In [21]:
# вычисляем выручку по формуле: количество * цена
sales['Revenue'] = sales['Quantity'] * sales['Price']

# сортируем заказы по выручке и оставляем первые 5
top_5_revenue = sales.sort_values('Revenue', ascending=False).head(5)
print(top_5_revenue[['OrderID', 'Product', 'Quantity', 'Price', 'Revenue']])

sales = sales.drop(columns=['Revenue'])
# удаляем столбец Revenue, чтобы в следующем задании работать с исходной таблицей

   OrderID Product  Quantity    Price   Revenue
3      104  Laptop         3  50000.0  150000.0
7      108   Phone         4  30000.0  120000.0
5      106  Laptop         2  50000.0  100000.0
1      102   Phone         2  30000.0   60000.0
8      109  Laptop         1  50000.0   50000.0


**3. Заполнить пропуски в Price значением, равным средней цене соответствующего продукта**

In [22]:
# используем исходную таблицу sales без Revenue
sales_filled = sales.copy()

# для каждого продукта заполняем пропуски средней ценой этого продукта
sales_filled['Price'] = sales_filled.groupby('Product')['Price'].transform(
    lambda x: x.fillna(x.mean())
)
print(sales_filled)

   OrderID Product  Quantity    Price  Region
0      101  Laptop         1  50000.0  Moscow
1      102   Phone         2  30000.0     SPb
2      103  Tablet         1  20000.0   Kazan
3      104  Laptop         3  50000.0  Moscow
4      105   Phone         1  30000.0     SPb
5      106  Laptop         2  50000.0   Kazan
6      107  Tablet         1  20000.0     NaN
7      108   Phone         4  30000.0     SPb
8      109  Laptop         1  50000.0  Moscow
9      110  Tablet         2  20000.0   Kazan


**4. Отсортировать данные по Region (по алфавиту), затем по Price (по убыванию)**

In [23]:
# используем таблицу после заполнения пропусков в Price из задания 3
sorted_sales = sales_filled.sort_values(
    ['Region', 'Price'],
    ascending=[True, False],
    na_position='last'
)
print(sorted_sales)


   OrderID Product  Quantity    Price  Region
5      106  Laptop         2  50000.0   Kazan
2      103  Tablet         1  20000.0   Kazan
9      110  Tablet         2  20000.0   Kazan
0      101  Laptop         1  50000.0  Moscow
3      104  Laptop         3  50000.0  Moscow
8      109  Laptop         1  50000.0  Moscow
1      102   Phone         2  30000.0     SPb
4      105   Phone         1  30000.0     SPb
7      108   Phone         4  30000.0     SPb
6      107  Tablet         1  20000.0     NaN


**5. Объединить таблицы, добавив столбец Weight_kg из products в sales, используя merge**

In [24]:
# объединяем таблицу после заполнения пропусков в Price с таблицей товаров по Product
merged_sales = sales_filled.merge(
    products[['Product', 'Weight_kg']],
    on='Product',
    how='left'
)
print(merged_sales)


   OrderID Product  Quantity    Price  Region  Weight_kg
0      101  Laptop         1  50000.0  Moscow        2.5
1      102   Phone         2  30000.0     SPb        0.2
2      103  Tablet         1  20000.0   Kazan        0.5
3      104  Laptop         3  50000.0  Moscow        2.5
4      105   Phone         1  30000.0     SPb        0.2
5      106  Laptop         2  50000.0   Kazan        2.5
6      107  Tablet         1  20000.0     NaN        0.5
7      108   Phone         4  30000.0     SPb        0.2
8      109  Laptop         1  50000.0  Moscow        2.5
9      110  Tablet         2  20000.0   Kazan        0.5
